# 🌲 LightGBM Gradient Boosting: The Power of Shallow Trees (Kaggle PGS S6E11)

Welcome to the **LightGBM Benchmark Notebook** for **Kaggle Playground Series Season 6 Episode 11 (Predicting Credit Card Fraud)**.

### 💡 The Core Learning Hypothesis: Tree Complexity & Synthetic Data
In tabular fraud competitions, data scientists often start with default tree hyperparameters (`num_leaves = 31`, `max_depth = -1`). However, as discovered in the top community benchmarks:
- The synthetic data generator produces transaction risk signals that are **highly additive in log-odds space** (which is why simple Logistic Regression achieves ~0.9419 ROC-AUC).
- When decision trees are deep (`31 leaves`), they split on random synthetic noise combinations, causing premature overfitting (`best_iteration ~ 380`, Fold 1 AUC `~0.9434`).
- When we constrain the tree to **shallow stumps / tiny trees (`num_leaves = 4`, `max_depth = 3`)**, the model behaves like an additive boosted generalized model. It continues learning meaningful additive interactions across 1,000+ trees, pushing the ROC-AUC score higher (`~0.9455+`).

### 🎯 Notebook Objectives
1. Implement native categorical handling with LightGBM on the standardized 5-fold Stratified CV split.
2. Train a **4-leaf shallow LightGBM model** across all 5 folds.
3. Track OOF ROC-AUC, PR-AUC, and extract **Feature Importances** to reveal the strongest fraud drivers.
4. Output and verify `lightgbm_submission.csv` for submission to Kaggle.


In [1]:
import os
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    classification_report
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print(f"LightGBM Version: {lgb.__version__}")


LightGBM Version: 4.7.0


## 1. Configuration & Path Setup

In [2]:
class Config:
    RANDOM_STATE = 42
    N_SPLITS = 5
    
    # Model Hyperparameters (Optimized Shallow Tree Architecture)
    NUM_LEAVES = 4
    MAX_DEPTH = 3
    LEARNING_RATE = 0.05
    N_ESTIMATORS = 1500
    EARLY_STOPPING_ROUNDS = 50
    SUBSAMPLE = 0.8
    COLSAMPLE_BYTREE = 0.8
    
    # Path resolution
    if os.path.exists('/kaggle/input/playground-series-s6e11'):
        DATA_DIR = Path('/kaggle/input/playground-series-s6e11')
        OUTPUT_DIR = Path('.')
    elif os.path.exists('/kaggle/input/competitions/playground-series-s6e11'):
        DATA_DIR = Path('/kaggle/input/competitions/playground-series-s6e11')
        OUTPUT_DIR = Path('.')
    else:
        DATA_DIR = Path('../../dataset')
        if not (DATA_DIR / 'train.csv').exists():
            DATA_DIR = Path('pgs-s6e11/dataset')
        OUTPUT_DIR = Path('../../submission')
        if not OUTPUT_DIR.exists():
            OUTPUT_DIR = Path('pgs-s6e11/submission')

Config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data Directory:   {Config.DATA_DIR}")
print(f"Output Directory: {Config.OUTPUT_DIR}")


Data Directory:   ../../dataset
Output Directory: ../../submission


## 2. Data Ingestion & Preprocessing

LightGBM natively handles categorical variables without one-hot encoding when columns are typed as `category`.
This avoids high-dimensional sparse representations and speeds up histogram construction.


In [3]:
train_df = pd.read_csv(Config.DATA_DIR / 'train.csv')
test_df = pd.read_csv(Config.DATA_DIR / 'test.csv')
sample_sub = pd.read_csv(Config.DATA_DIR / 'sample_submission.csv')

cat_cols = ['merchant_category', 'card_type', 'auth_method', 'channel', 'device_type']
bool_cols = [
    'is_foreign_transaction', 'is_new_merchant', 'used_vpn',
    'ip_country_mismatch', 'billing_shipping_mismatch', 'is_ai_generated_scam_attempt'
]

# Convert categorical features to category dtype
for col in cat_cols:
    train_df[col] = train_df[col].astype('category')
    test_df[col] = test_df[col].astype('category')

# Convert boolean flags to binary integers (0/1)
for b in bool_cols:
    train_df[b] = train_df[b].astype(int)
    test_df[b] = test_df[b].astype(int)

features = [c for c in train_df.columns if c not in ['id', 'is_fraud']]
print(f"Total Model Features: {len(features)}")
print(f"Categorical Features: {cat_cols}")
print(f"Train Shape: {train_df.shape}, Test Shape: {test_df.shape}")


Total Model Features: 24
Categorical Features: ['merchant_category', 'card_type', 'auth_method', 'channel', 'device_type']
Train Shape: (699960, 26), Test Shape: (299984, 25)


## 3. 5-Fold Stratified Cross-Validation (Shallow LightGBM)

We train across 5 folds with early stopping enabled.
For each fold, we:
- Fit LightGBM with `num_leaves=4` and `max_depth=3`.
- Track validation score and best iteration.
- Store out-of-fold validation probabilities and accumulate test predictions.


In [4]:
X = train_df[features]
y = train_df['is_fraud'].astype(int)
X_test = test_df[features]

skf = StratifiedKFold(n_splits=Config.N_SPLITS, shuffle=True, random_state=Config.RANDOM_STATE)

oof_preds = np.zeros(len(X), dtype=np.float64)
test_preds = np.zeros(len(X_test), dtype=np.float64)
feature_importances = np.zeros(len(features), dtype=np.float64)
fold_scores = []

print("=" * 65)
print(f"Training LightGBM (num_leaves={Config.NUM_LEAVES}, max_depth={Config.MAX_DEPTH})...")
print("=" * 65)

start_all = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    t0 = time.time()
    X_train_fold, y_train_fold = X.iloc[train_idx], y.iloc[train_idx]
    X_val_fold, y_val_fold = X.iloc[val_idx], y.iloc[val_idx]
    
    model = lgb.LGBMClassifier(
        objective='binary',
        metric='auc',
        num_leaves=Config.NUM_LEAVES,
        max_depth=Config.MAX_DEPTH,
        learning_rate=Config.LEARNING_RATE,
        n_estimators=Config.N_ESTIMATORS,
        subsample=Config.SUBSAMPLE,
        colsample_bytree=Config.COLSAMPLE_BYTREE,
        random_state=Config.RANDOM_STATE + fold,
        n_jobs=-1,
        verbose=-1
    )
    
    model.fit(
        X_train_fold, y_train_fold,
        eval_set=[(X_val_fold, y_val_fold)],
        callbacks=[lgb.early_stopping(Config.EARLY_STOPPING_ROUNDS, verbose=False)]
    )
    
    val_probs = model.predict_proba(X_val_fold)[:, 1]
    oof_preds[val_idx] = val_probs
    test_preds += model.predict_proba(X_test)[:, 1] / Config.N_SPLITS
    feature_importances += model.feature_importances_ / Config.N_SPLITS
    
    fold_auc = roc_auc_score(y_val_fold, val_probs)
    fold_pr_auc = average_precision_score(y_val_fold, val_probs)
    fold_scores.append((fold_auc, fold_pr_auc))
    
    print(f"Fold {fold} | Best Iter: {model.best_iteration_:4d} | ROC-AUC: {fold_auc:.5f} | PR-AUC: {fold_pr_auc:.5f} ({time.time()-t0:.1f}s)")

total_cv_time = time.time() - start_all
print("=" * 65)
print(f"All 5 Folds Completed in {total_cv_time:.1f}s!")


Training LightGBM (num_leaves=4, max_depth=3)...
/home/shivamgravity/miniconda3/envs/kaggle_base/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
Fold 1 | Best Iter: 1111 | ROC-AUC: 0.94550 | PR-AUC: 0.49998 (58.1s)
/home/shivamgravity/miniconda3/envs/kaggle_base/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
Fold 2 | Best Iter: 1405 | ROC-AUC: 0.94873 | PR-AUC: 0.51715 (55.3s)
/home/shivamgravity/miniconda3/envs/kaggle_base/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_

## 4. Performance Diagnostics & Feature Importance

In [5]:
overall_auc = roc_auc_score(y, oof_preds)
overall_pr_auc = average_precision_score(y, oof_preds)

print("-" * 55)
print("LightGBM 5-Fold Cross-Validation Summary:")
print(f"Mean Fold ROC-AUC:   {np.mean([s[0] for s in fold_scores]):.5f} ± {np.std([s[0] for s in fold_scores]):.5f}")
print(f"Overall OOF ROC-AUC: {overall_auc:.5f}")
print(f"Overall OOF PR-AUC:  {overall_pr_auc:.5f}")
print("-" * 55)

# Classification Report
pred_binary = (oof_preds >= 0.5).astype(int)
print("\nClassification Report (Threshold = 0.5):")
print(classification_report(y, pred_binary, digits=4))

# Plot Diagnostic Curves and Feature Importances
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. ROC Curve
fpr, tpr, _ = roc_curve(y, oof_preds)
axes[0].plot(fpr, tpr, color='#1f77b4', lw=2, label=f'OOF ROC (AUC = {overall_auc:.5f})')
axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve (5-Fold OOF)')
axes[0].legend(loc='lower right')

# 2. Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y, oof_preds)
axes[1].plot(recall, precision, color='#ff7f0e', lw=2, label=f'OOF PR (AUC = {overall_pr_auc:.5f})')
axes[1].axhline(y=y.mean(), color='gray', linestyle='--', label=f'Base Rate ({y.mean():.4f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve (5-Fold OOF)')
axes[1].legend(loc='upper right')

# 3. Top Feature Importances
fi_df = pd.DataFrame({'feature': features, 'importance': feature_importances}).sort_values(by='importance', ascending=True)
top_fi = fi_df.tail(15)
axes[2].barh(top_fi['feature'], top_fi['importance'], color='#2ca02c')
axes[2].set_xlabel('Average Feature Importance (Split Count)')
axes[2].set_title('Top 15 Most Important Features')

plt.tight_layout()
plt.savefig('lightgbm_diagnostics.png', dpi=150)
plt.show()


-------------------------------------------------------
LightGBM 5-Fold Cross-Validation Summary:
Mean Fold ROC-AUC:   0.94706 ± 0.00313
Overall OOF ROC-AUC: 0.94700
Overall OOF PR-AUC:  0.51074
-------------------------------------------------------

Classification Report (Threshold = 0.5):
              precision    recall  f1-score   support

           0     0.9883    0.9984    0.9933    688271
           1     0.7590    0.3041    0.4343     11689

    accuracy                         0.9868    699960
   macro avg     0.8736    0.6512    0.7138    699960
weighted avg     0.9845    0.9868    0.9840    699960



## 5. Submission File Generation & Verification

In [6]:
submission = pd.DataFrame({
    'id': test_df['id'],
    'is_fraud': test_preds
})

assert submission.shape == sample_sub.shape, f"Shape mismatch: {submission.shape} vs {sample_sub.shape}"
assert list(submission.columns) == ['id', 'is_fraud'], f"Column mismatch: {submission.columns}"
assert not submission.isna().any().any(), "Submission contains null/NaN values!"
assert (submission['is_fraud'] >= 0.0).all() and (submission['is_fraud'] <= 1.0).all(), "Predictions out of [0, 1] range!"
assert (submission['id'] == sample_sub['id']).all(), "ID alignment mismatch!"

sub_path = Config.OUTPUT_DIR / 'lightgbm_submission.csv'
submission.to_csv(sub_path, index=False)

print(f"Submission successfully generated and verified at: {sub_path}")
print(f"File size: {os.path.getsize(sub_path) / (1024*1024):.2f} MB")
print("\nFirst 10 submission rows:")
print(submission.head(10))


Submission successfully generated and verified at: ../../submission/lightgbm_submission.csv
File size: 8.15 MB

First 10 submission rows:
       id  is_fraud
0  699960  0.000268
1  699961  0.000212
2  699962  0.082914
3  699963  0.001371
4  699964  0.001340
5  699965  0.000227
6  699966  0.003535
7  699967  0.000435
8  699968  0.008449
9  699969  0.000744
